# Gridiron Intelligence: Supabase Upserts Pipeline

This notebook prepares Supabase tables and upsert payloads for the planned agent workflow:

1. **Player resolution + SQL retrieval** (`player_master`, scouting, model predictions)
2. **Vector retrieval** for historical factoids (`factoid_vectors`)
3. **Web intel summarization** from DuckDuckGo + Gemini 2.5-lite (`news_chunks`, `news_summaries`)
4. **Final LLM synthesis** from all context

Defaults are intentionally placeholders so credentials and deployment-specific values can be entered later.

In [14]:
from __future__ import annotations

import ast
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from supabase import create_client, Client
except Exception:
    create_client = None
    Client = Any

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [15]:
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "modeling_datasets").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/modeling_datasets")

PROJECT_ROOT = resolve_project_root()
FINAL_DIR = PROJECT_ROOT / "data" / "modeling_datasets" / "final"
RECRUITS_DIR = PROJECT_ROOT / "data" / "modeling_datasets" / "recruits"
MATCH_DIR = PROJECT_ROOT / "data" / "name_matching" / "recruit_college"
MODELS_DIR = FINAL_DIR / "models" / "xgboost_positional"
MODELS_SUMMARY_DIR = MODELS_DIR / "summaries"

factoid_vectors_candidates = [
    FINAL_DIR / "factoids" / "gridiron_scouting_factoids_supabase_vectors.csv",
    FINAL_DIR / "gridiron_scouting_factoids_supabase_vectors.csv",
]
factoid_vectors_path = next((p for p in factoid_vectors_candidates if p.exists()), factoid_vectors_candidates[0])

PATHS = {
    "master_recruits": RECRUITS_DIR / "master_recruits_2015_2028.csv",
    "master_scouting": RECRUITS_DIR / "master_scouting_reports_2015_2028.csv",
    "all_matches": MATCH_DIR / "all_matches_combined.csv",
    "xgb_score": MODELS_SUMMARY_DIR / "xgb_all_positions_predictions_2022_2028_calibrated.csv",
    "xgb_class": MODELS_SUMMARY_DIR / "xgb_class_all_positions_predictions_2022_2028_raw.csv",
    "factoid_vectors": factoid_vectors_path,
}

for name, p in PATHS.items():
    print(f"{name:16s} -> {p} | exists={p.exists()}")

master_recruits  -> x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits\master_recruits_2015_2028.csv | exists=True
master_scouting  -> x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits\master_scouting_reports_2015_2028.csv | exists=True
all_matches      -> x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\name_matching\recruit_college\all_matches_combined.csv | exists=True
xgb_score        -> x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\final\models\xgboost_positional\summaries\xgb_all_positions_predictions_2022_2028_calibrated.csv | exists=True
xgb_class        -> x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\final\models\xgboost_positional\summaries\xgb_class_all_positions_predictions_2022_2028_raw.csv | exists=True
factoid_vectors  -> x:\My Files\Courses\DSBA

## Supabase Configuration (Placeholders)

Enter values directly here or export env vars before running.

- `SUPABASE_URL`
- `SUPABASE_SERVICE_ROLE_KEY`
- optional schema/table names

In [16]:
ENV_PATH = PROJECT_ROOT / "SUPABASE.env"
if load_dotenv is not None and ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=False)
    print(f"Loaded environment values from: {ENV_PATH}")
elif not ENV_PATH.exists():
    print(f"SUPABASE.env not found at: {ENV_PATH}")
else:
    print("python-dotenv not available; using current process environment only")

CONFIG = {
    "SUPABASE_URL": os.getenv("SUPABASE_URL", "<ENTER_SUPABASE_URL>"),
    "SUPABASE_ANON_KEY": os.getenv("SUPABASE_ANON_KEY", "<ENTER_SUPABASE_ANON_KEY>"),
    "SUPABASE_SERVICE_ROLE_KEY": os.getenv("SUPABASE_SERVICE_ROLE_KEY", "<ENTER_SUPABASE_SERVICE_ROLE_KEY>"),
    "SCHEMA": os.getenv("SUPABASE_SCHEMA", "public"),
    "DRY_RUN": True,
    "UPSERT_BATCH_SIZE": 500,
    "MAX_RETRIES": 3,
    "SLEEP_SECONDS": 0.75,
    "MODEL_VERSION": "xgb_positional_v1",
    "EMBEDDING_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
    "FACTOID_VECTOR_DIM": 384,
    "DB_USER": os.getenv("DB_USER", "<ENTER_DB_USER>"),
    "DB_PASSWORD": os.getenv("DB_PASSWORD", "<ENTER_DB_PASSWORD>"),
    "DB_HOST": os.getenv("DB_HOST", "<ENTER_DB_HOST>"),
    "DB_PORT": os.getenv("DB_PORT", "6543"),
    "DB_NAME": os.getenv("DB_NAME", "postgres"),
}

CONFIG["DB_DSN"] = (
    f"postgresql://{CONFIG['DB_USER']}:{CONFIG['DB_PASSWORD']}@"
    f"{CONFIG['DB_HOST']}:{CONFIG['DB_PORT']}/{CONFIG['DB_NAME']}"
)

TABLES = {
    "player_master": "gi_player_master",
    "player_link_bridge": "gi_player_link_bridge",
    "scouting_features": "gi_scouting_report_features",
    "pred_score": "gi_model_prediction_score",
    "pred_threshold": "gi_model_prediction_thresholds",
    "factoid_vectors": "gi_factoid_vectors",
    "news_chunks": "gi_news_chunks",
    "news_summaries": "gi_news_summaries",
}

print("Supabase target tables:")
for k, v in TABLES.items():
    print(f"  {k:18s} -> {v}")

print("\nCredential status:")
print(f"  SUPABASE_URL set: {CONFIG['SUPABASE_URL'].startswith('http')}")
print(f"  SERVICE_ROLE set: {not CONFIG['SUPABASE_SERVICE_ROLE_KEY'].startswith('<ENTER_')}")
print(f"  DB host set: {not CONFIG['DB_HOST'].startswith('<ENTER_')}")

Loaded environment values from: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\SUPABASE.env
Supabase target tables:
  player_master      -> gi_player_master
  player_link_bridge -> gi_player_link_bridge
  scouting_features  -> gi_scouting_report_features
  pred_score         -> gi_model_prediction_score
  pred_threshold     -> gi_model_prediction_thresholds
  factoid_vectors    -> gi_factoid_vectors
  news_chunks        -> gi_news_chunks
  news_summaries     -> gi_news_summaries

Credential status:
  SUPABASE_URL set: True
  SERVICE_ROLE set: True
  DB host set: True


In [17]:
db_password = str(CONFIG.get("DB_PASSWORD", ""))
masked = "*" * min(len(db_password), 8) if db_password and not db_password.startswith("<ENTER_") else "<unset>"
print(f"DB_PASSWORD configured: {masked}")

DB_PASSWORD configured: ********


In [18]:
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

try:
    conn = psycopg2.connect(
        user=CONFIG['DB_USER'],
        password=CONFIG['DB_PASSWORD'], 
        host=CONFIG['DB_HOST'],
        port=CONFIG['DB_PORT'],
        dbname=CONFIG['DB_NAME']
    )
    print("Success!")
    conn.close()
except Exception as e:
    print(f"Auth failed: {e}")

Success!


## Suggested SQL DDL (Run in Supabase SQL editor once)

This notebook performs upserts via PostgREST. Ensure tables exist first.

In [19]:
DDL_SQL = f"""
create extension if not exists vector;

create table if not exists {TABLES['player_master']} (
  recruit_id text primary key,
  sports_ref_id text,
  player_name text,
  year int,
  position text,
  rating numeric,
  height_raw text,
  weight_raw text,
  height_inches numeric,
  weight_lbs numeric,
  committed_to text,
  high_school text,
  city text,
  state text,
  updated_at timestamptz default now()
);

alter table {TABLES['player_master']} add column if not exists height_raw text;
alter table {TABLES['player_master']} add column if not exists weight_raw text;
alter table {TABLES['player_master']} add column if not exists height_inches numeric;
alter table {TABLES['player_master']} add column if not exists weight_lbs numeric;

create table if not exists {TABLES['player_link_bridge']} (
  recruit_id text not null,
  sports_ref_id text not null,
  match_strategy text,
  name_score numeric,
  team_match numeric,
  year_diff numeric,
  is_primary boolean default true,
  updated_at timestamptz default now(),
  primary key (recruit_id, sports_ref_id)
);

create table if not exists {TABLES['scouting_features']} (
  recruit_id text primary key,
  hs_athletic_background text,
  scouting_json jsonb,
  updated_at timestamptz default now()
);

create table if not exists {TABLES['pred_score']} (
  recruit_id text not null,
  sports_ref_id text,
  position_group text,
  predictive_score_0_100 numeric,
  contrib_tier_raw text,
  model_version text not null,
  as_of_date date not null,
  updated_at timestamptz default now(),
  primary key (recruit_id, model_version, as_of_date)
);

alter table {TABLES['pred_score']} add column if not exists contrib_tier_raw text;

create table if not exists {TABLES['pred_threshold']} (
  recruit_id text not null,
  sports_ref_id text,
  position_group text,
  prob_ge20 numeric,
  prob_ge50 numeric,
  prob_ge80 numeric,
  odds_ge20 numeric,
  odds_ge50 numeric,
  odds_ge80 numeric,
  contrib_tier_raw text,
  model_version text not null,
  as_of_date date not null,
  updated_at timestamptz default now(),
  primary key (recruit_id, model_version, as_of_date)
);

create table if not exists {TABLES['factoid_vectors']} (
  id text primary key,
  position text,
  factoid_type text,
  factoid_text text,
  tags text,
  search_text text,
  embedding vector({CONFIG['FACTOID_VECTOR_DIM']}),
  embedding_model text,
  embedding_device text,
  created_at timestamptz
);

create index if not exists ix_{TABLES['factoid_vectors']}_position on {TABLES['factoid_vectors']}(position);
create index if not exists ix_{TABLES['factoid_vectors']}_type on {TABLES['factoid_vectors']}(factoid_type);

-- Optional workflow tables for web/news context
create table if not exists {TABLES['news_chunks']} (
  chunk_id text primary key,
  recruit_id text,
  sports_ref_id text,
  source text,
  title text,
  url text,
  chunk_text text,
  published_at timestamptz,
  embedding vector({CONFIG['FACTOID_VECTOR_DIM']}),
  embedding_model text,
  created_at timestamptz default now()
);

create table if not exists {TABLES['news_summaries']} (
  summary_id text primary key,
  recruit_id text,
  sports_ref_id text,
  llm_model text,
  prompt_version text,
  summary_text text,
  extracted_stats_json jsonb,
  created_at timestamptz default now()
);
"""

print(DDL_SQL[:4000])


create extension if not exists vector;

create table if not exists gi_player_master (
  recruit_id text primary key,
  sports_ref_id text,
  player_name text,
  year int,
  position text,
  rating numeric,
  height_raw text,
  weight_raw text,
  height_inches numeric,
  weight_lbs numeric,
  committed_to text,
  high_school text,
  city text,
  state text,
  updated_at timestamptz default now()
);

alter table gi_player_master add column if not exists height_raw text;
alter table gi_player_master add column if not exists weight_raw text;
alter table gi_player_master add column if not exists height_inches numeric;
alter table gi_player_master add column if not exists weight_lbs numeric;

create table if not exists gi_player_link_bridge (
  recruit_id text not null,
  sports_ref_id text not null,
  match_strategy text,
  name_score numeric,
  team_match numeric,
  year_diff numeric,
  is_primary boolean default true,
  updated_at timestamptz default now(),
  primary key (recruit_id, spo

## One-Time Migration/Backfill (Run if `pred_score` upsert errors on `contrib_tier_raw`)

Use this SQL in Supabase SQL Editor to ensure schema + historical data are aligned before rerunning upserts.

In [ ]:
MIGRATION_SQL = f"""
alter table {TABLES['pred_score']} add column if not exists contrib_tier_raw text;

update {TABLES['pred_score']} ps
set contrib_tier_raw = pt.contrib_tier_raw
from {TABLES['pred_threshold']} pt
where ps.recruit_id = pt.recruit_id
  and ps.model_version = pt.model_version
  and ps.as_of_date = pt.as_of_date
  and ps.contrib_tier_raw is null
  and pt.contrib_tier_raw is not null;

notify pgrst, 'reload schema';
"""

print(MIGRATION_SQL)

## Load Source Data

In [20]:
def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

df_recruits = safe_read_csv(PATHS['master_recruits'])
df_scouting = safe_read_csv(PATHS['master_scouting'])
df_matches = safe_read_csv(PATHS['all_matches'])
df_pred_score = safe_read_csv(PATHS['xgb_score'])
df_pred_class = safe_read_csv(PATHS['xgb_class'])
df_factoid_vectors = safe_read_csv(PATHS['factoid_vectors'])

print('Loaded dataframes:')
for name, df in [
    ('recruits', df_recruits),
    ('scouting', df_scouting),
    ('matches', df_matches),
    ('pred_score', df_pred_score),
    ('pred_class', df_pred_class),
    ('factoid_vectors', df_factoid_vectors),
]:
    print(f"  {name:14s}: {len(df):>7,} rows | cols={len(df.columns)}")

Loaded dataframes:
  recruits      :  41,926 rows | cols=13
  scouting      :  41,925 rows | cols=79
  matches       :  25,353 rows | cols=15
  pred_score    :  14,392 rows | cols=12
  pred_class    :  14,392 rows | cols=14
  factoid_vectors:     336 rows | cols=10


In [21]:
# Standardize ids
for df in [df_recruits, df_scouting, df_matches, df_pred_score, df_pred_class]:
    if df.empty:
        continue
    
    # Canonical id for Supabase tables
    if 'recruit_id' not in df.columns and 'player_id' in df.columns:
        df['recruit_id'] = df['player_id']
    
    if 'recruit_id' in df.columns:
        df['recruit_id'] = df['recruit_id'].astype(str).str.strip()

if not df_pred_score.empty and 'player_id' in df_pred_score.columns:
    df_pred_score['recruit_id'] = df_pred_score['player_id'].astype(str).str.strip()

if not df_pred_class.empty and 'player_id' in df_pred_class.columns:
    df_pred_class['recruit_id'] = df_pred_class['player_id'].astype(str).str.strip()

if not df_matches.empty:
    if 'recruit_id' in df_matches.columns:
        df_matches['recruit_id'] = df_matches['recruit_id'].astype(str).str.strip()
    if 'sports_ref_id' in df_matches.columns:
        df_matches['sports_ref_id'] = df_matches['sports_ref_id'].astype(str).str.strip()

## Build Upsert Payloads

In [22]:
# Bridge: recruit_id -> sports_ref_id
bridge_cols = [c for c in ['recruit_id', 'sports_ref_id', 'match_strategy', 'name_score', 'team_match', 'year_diff'] if c in df_matches.columns]
bridge_df = df_matches[bridge_cols].dropna(subset=['recruit_id'], how='any').copy() if bridge_cols else pd.DataFrame(columns=['recruit_id', 'sports_ref_id'])
if 'sports_ref_id' in bridge_df.columns:
    bridge_df = bridge_df[bridge_df['sports_ref_id'].astype(str).str.strip() != '']
bridge_df = bridge_df.drop_duplicates(subset=['recruit_id', 'sports_ref_id'])

# Canonicalize recruits/scouting ids in case source files use player_id
if not df_recruits.empty and 'recruit_id' not in df_recruits.columns and 'player_id' in df_recruits.columns:
    df_recruits = df_recruits.copy()
    df_recruits['recruit_id'] = df_recruits['player_id'].astype(str).str.strip()

if not df_scouting.empty and 'recruit_id' not in df_scouting.columns and 'player_id' in df_scouting.columns:
    df_scouting = df_scouting.copy()
    df_scouting['recruit_id'] = df_scouting['player_id'].astype(str).str.strip()

def parse_height_to_inches(value: Any) -> float | None:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip()
    if not text:
        return None

    import re
    match = re.match(r"^(\d+)\s*'\s*(\d+(?:\.\d+)?)\"?$", text)
    if match:
        feet = float(match.group(1))
        inches = float(match.group(2))
        return feet * 12 + inches

    num = pd.to_numeric(text, errors='coerce')
    if pd.isna(num):
        return None
    return float(num)

def parse_weight_to_lbs(value: Any) -> float | None:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip().lower()
    if not text:
        return None
    text = text.replace('lbs', '').replace('lb', '').strip()
    num = pd.to_numeric(text, errors='coerce')
    if pd.isna(num):
        return None
    return float(num)

# Player master payload
master_cols = [c for c in ['recruit_id', 'name', 'year', 'position', 'rating', 'height', 'weight', 'committed_to', 'high_school', 'city', 'state'] if c in df_recruits.columns]
if 'recruit_id' not in master_cols:
    raise KeyError("Expected recruit_id (or player_id) in master recruits file")
player_master_df = df_recruits[master_cols].copy()
player_master_df = player_master_df.rename(columns={'name': 'player_name', 'height': 'height_raw', 'weight': 'weight_raw'})

if 'height_raw' not in player_master_df.columns:
    player_master_df['height_raw'] = None
if 'weight_raw' not in player_master_df.columns:
    player_master_df['weight_raw'] = None

player_master_df['height_inches'] = player_master_df['height_raw'].apply(parse_height_to_inches)
player_master_df['weight_lbs'] = player_master_df['weight_raw'].apply(parse_weight_to_lbs)

if not bridge_df.empty:
    sr_map = bridge_df.groupby('recruit_id', as_index=False)['sports_ref_id'].first()
    player_master_df = player_master_df.merge(sr_map, on='recruit_id', how='left')
else:
    player_master_df['sports_ref_id'] = None

player_master_df['updated_at'] = datetime.now(timezone.utc).isoformat()
player_master_df = player_master_df.drop_duplicates(subset=['recruit_id'])

# Scouting features payload
scouting_payload_df = pd.DataFrame(columns=['recruit_id', 'hs_athletic_background', 'scouting_json', 'updated_at'])
if not df_scouting.empty and 'recruit_id' in df_scouting.columns:
    keep_cols = [c for c in df_scouting.columns if c.startswith('flag_') or c.startswith('skill_')]
    core_cols = [c for c in ['recruit_id', 'hs_athletic_background'] if c in df_scouting.columns]
    tmp = df_scouting[core_cols + keep_cols].copy()
    tmp = tmp.drop_duplicates(subset=['recruit_id'])
    tmp['scouting_json'] = tmp[keep_cols].to_dict(orient='records') if keep_cols else [{} for _ in range(len(tmp))]
    scouting_payload_df = tmp[['recruit_id'] + [c for c in ['hs_athletic_background'] if c in tmp.columns] + ['scouting_json']].copy()
    if 'hs_athletic_background' not in scouting_payload_df.columns:
        scouting_payload_df['hs_athletic_background'] = None
    scouting_payload_df['updated_at'] = datetime.now(timezone.utc).isoformat()

# Prediction score payload (regressor outputs + regressor-derived career designation tier)
pred_score_payload = pd.DataFrame(columns=['recruit_id'])
if not df_pred_score.empty and 'recruit_id' in df_pred_score.columns:
    tier_source_col = None
    for candidate in ['contrib_tier_raw', 'y_pred_tier', 'y_pred_tier_raw']:
        if candidate in df_pred_score.columns:
            tier_source_col = candidate
            break

    base_cols = [c for c in ['recruit_id', 'position_group', 'predictive_score_0_100'] if c in df_pred_score.columns]
    if tier_source_col is not None and tier_source_col not in base_cols:
        base_cols.append(tier_source_col)

    pred_score_payload = df_pred_score[base_cols].copy()
    if tier_source_col is not None and tier_source_col != 'contrib_tier_raw':
        pred_score_payload = pred_score_payload.rename(columns={tier_source_col: 'contrib_tier_raw'})
    if 'contrib_tier_raw' not in pred_score_payload.columns:
        pred_score_payload['contrib_tier_raw'] = None

    pred_score_payload['predictive_score_0_100'] = pd.to_numeric(pred_score_payload.get('predictive_score_0_100'), errors='coerce')
    if not bridge_df.empty:
        pred_score_payload = pred_score_payload.merge(sr_map, on='recruit_id', how='left')
    else:
        pred_score_payload['sports_ref_id'] = None
    pred_score_payload['model_version'] = CONFIG['MODEL_VERSION']
    pred_score_payload['as_of_date'] = datetime.now(timezone.utc).date().isoformat()
    pred_score_payload['updated_at'] = datetime.now(timezone.utc).isoformat()
    pred_score_payload = pred_score_payload.drop_duplicates(subset=['recruit_id', 'model_version', 'as_of_date'])

# Prediction threshold payload (classification thresholds only)
pred_thr_payload = pd.DataFrame(columns=['recruit_id'])
if not df_pred_class.empty and 'recruit_id' in df_pred_class.columns:
    class_map = {
        'contrib_prob_ge20_raw': 'prob_ge20',
        'contrib_prob_ge50_raw': 'prob_ge50',
        'contrib_prob_ge80_raw': 'prob_ge80',
        'contrib_odds_ge20_raw': 'odds_ge20',
        'contrib_odds_ge50_raw': 'odds_ge50',
        'contrib_odds_ge80_raw': 'odds_ge80',
    }
    base_cols = [c for c in ['recruit_id', 'position_group'] if c in df_pred_class.columns]
    existing_map_cols = [c for c in class_map.keys() if c in df_pred_class.columns]
    pred_thr_payload = df_pred_class[base_cols + existing_map_cols].rename(columns=class_map).copy()

    for c in ['prob_ge20', 'prob_ge50', 'prob_ge80', 'odds_ge20', 'odds_ge50', 'odds_ge80']:
        if c in pred_thr_payload.columns:
            pred_thr_payload[c] = pd.to_numeric(pred_thr_payload[c], errors='coerce')

    if not bridge_df.empty:
        pred_thr_payload = pred_thr_payload.merge(sr_map, on='recruit_id', how='left')
    else:
        pred_thr_payload['sports_ref_id'] = None

    pred_thr_payload['model_version'] = CONFIG['MODEL_VERSION']
    pred_thr_payload['as_of_date'] = datetime.now(timezone.utc).date().isoformat()
    pred_thr_payload['updated_at'] = datetime.now(timezone.utc).isoformat()
    pred_thr_payload = pred_thr_payload.drop_duplicates(subset=['recruit_id', 'model_version', 'as_of_date'])

# Fill contrib_tier_raw in threshold payload from regressor-derived tier
if not pred_thr_payload.empty:
    if 'contrib_tier_raw' not in pred_thr_payload.columns:
        pred_thr_payload['contrib_tier_raw'] = None
    if not pred_score_payload.empty and 'contrib_tier_raw' in pred_score_payload.columns:
        tier_lookup = pred_score_payload[['recruit_id', 'contrib_tier_raw']].drop_duplicates(subset=['recruit_id'])
        pred_thr_payload = pred_thr_payload.merge(
            tier_lookup.rename(columns={'contrib_tier_raw': 'contrib_tier_raw_from_score'}),
            on='recruit_id',
            how='left'
        )
        pred_thr_payload['contrib_tier_raw'] = pred_thr_payload['contrib_tier_raw_from_score'].where(
            pred_thr_payload['contrib_tier_raw_from_score'].notna(),
            pred_thr_payload['contrib_tier_raw']
        )
        pred_thr_payload = pred_thr_payload.drop(columns=['contrib_tier_raw_from_score'])

print(f"player_master payload rows:      {len(player_master_df):,}")
print(f"player_link_bridge payload rows: {len(bridge_df):,}")
print(f"scouting_features payload rows:  {len(scouting_payload_df):,}")
print(f"pred_score payload rows:         {len(pred_score_payload):,}")
print(f"pred_threshold payload rows:     {len(pred_thr_payload):,}")
if not player_master_df.empty:
    print("\nplayer_master measurement preview:")
    print(player_master_df[['recruit_id', 'height_raw', 'height_inches', 'weight_raw', 'weight_lbs']].head(5).to_string(index=False))

player_master payload rows:      41,926
player_link_bridge payload rows: 25,353
scouting_features payload rows:  41,925
pred_score payload rows:         14,392
pred_threshold payload rows:     14,392

player_master measurement preview:
recruit_id height_raw  height_inches weight_raw  weight_lbs
 201500001    6' 2.5"           74.5    313 lbs       313.0
 201500002    6' 5.5"           77.5    275 lbs       275.0
 201500003      6' 4"           76.0    250 lbs       250.0
 201500004      6' 1"           73.0    190 lbs       190.0
 201500005      6' 2"           74.0    201 lbs       201.0


In [23]:
# Factoid vectors payload
factoid_payload = pd.DataFrame()
if not df_factoid_vectors.empty:
    required = ['id', 'position', 'factoid_type', 'factoid_text', 'tags', 'search_text', 'embedding']
    missing = [c for c in required if c not in df_factoid_vectors.columns]
    if missing:
        raise ValueError(f"Missing required vector columns: {missing}")

    factoid_payload = df_factoid_vectors.copy()

    def parse_embedding(x: Any) -> list[float]:
        if isinstance(x, list):
            arr = x
        else:
            txt = str(x).strip()
            try:
                arr = json.loads(txt)
            except Exception:
                arr = ast.literal_eval(txt)
        return [float(v) for v in arr]

    factoid_payload['embedding'] = factoid_payload['embedding'].apply(parse_embedding)

    dims = factoid_payload['embedding'].apply(len).value_counts()
    print('Embedding dimensions distribution:')
    print(dims)

    if len(dims.index) > 1:
        raise ValueError('Inconsistent vector dimensions in factoid embeddings')

    actual_dim = int(dims.index[0])
    CONFIG['FACTOID_VECTOR_DIM'] = actual_dim

    if 'created_at' not in factoid_payload.columns:
        factoid_payload['created_at'] = datetime.now(timezone.utc).isoformat()

print(f"factoid_vectors payload rows: {len(factoid_payload):,}")

Embedding dimensions distribution:
embedding
384    336
Name: count, dtype: int64
factoid_vectors payload rows: 336


## Supabase Upsert Helpers

In [24]:
def get_supabase_client() -> Client | None:
    if create_client is None:
        print('Install dependency: pip install supabase')
        return None
    if CONFIG['SUPABASE_URL'].startswith('<ENTER_') or CONFIG['SUPABASE_SERVICE_ROLE_KEY'].startswith('<ENTER_'):
        print('Supabase credentials are placeholders. Update CONFIG to run live upserts.')
        return None
    return create_client(CONFIG['SUPABASE_URL'], CONFIG['SUPABASE_SERVICE_ROLE_KEY'])

def chunk_records(records: list[dict], batch_size: int) -> list[list[dict]]:
    return [records[i:i + batch_size] for i in range(0, len(records), batch_size)]

def sanitize_for_json(value: Any) -> Any:
    if isinstance(value, dict):
        return {k: sanitize_for_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_for_json(v) for v in value]
    if isinstance(value, tuple):
        return [sanitize_for_json(v) for v in value]
    if value is None:
        return None
    if isinstance(value, (np.floating, float)) and (np.isnan(value) or np.isinf(value)):
        return None
    if isinstance(value, (np.integer, int, str, bool)):
        return value
    if pd.isna(value):
        return None
    return value

def upsert_dataframe(
    client: Client | None,
    table_name: str,
    df: pd.DataFrame,
    on_conflict: str,
    dry_run: bool = True,
) -> None:
    if df.empty:
        print(f"[{table_name}] No rows to upsert")
        return

    records = df.replace({np.nan: None}).to_dict(orient='records')
    records = [sanitize_for_json(r) for r in records]
    batches = chunk_records(records, CONFIG['UPSERT_BATCH_SIZE'])

    print(f"[{table_name}] rows={len(records):,}, batches={len(batches)}, on_conflict={on_conflict}")
    if dry_run or client is None:
        print(f"[{table_name}] DRY_RUN enabled (or missing client): no write executed")
        return

    for i, batch in enumerate(batches, start=1):
        last_error = None
        for attempt in range(1, CONFIG['MAX_RETRIES'] + 1):
            try:
                client.table(table_name).upsert(batch, on_conflict=on_conflict).execute()
                last_error = None
                break
            except Exception as exc:
                last_error = exc
                time.sleep(CONFIG['SLEEP_SECONDS'] * attempt)
        if last_error is not None:
            raise RuntimeError(f"Upsert failed for {table_name}, batch {i}: {last_error}")

    print(f"[{table_name}] Upsert complete")

## Execute Upsert Sequence

Order mirrors workflow dependencies: player entities first, then predictions and vectors.

In [25]:
CONFIG['DRY_RUN'] = False
print(f"DRY_RUN set to: {CONFIG['DRY_RUN']}")

DRY_RUN set to: False


In [26]:
sb = get_supabase_client()

upsert_dataframe(
    client=sb,
    table_name=TABLES['player_master'],
    df=player_master_df,
    on_conflict='recruit_id',
    dry_run=CONFIG['DRY_RUN'],
)

upsert_dataframe(
    client=sb,
    table_name=TABLES['player_link_bridge'],
    df=bridge_df.assign(is_primary=True, updated_at=datetime.now(timezone.utc).isoformat()),
    on_conflict='recruit_id,sports_ref_id',
    dry_run=CONFIG['DRY_RUN'],
)

upsert_dataframe(
    client=sb,
    table_name=TABLES['scouting_features'],
    df=scouting_payload_df,
    on_conflict='recruit_id',
    dry_run=CONFIG['DRY_RUN'],
)

upsert_dataframe(
    client=sb,
    table_name=TABLES['pred_score'],
    df=pred_score_payload,
    on_conflict='recruit_id,model_version,as_of_date',
    dry_run=CONFIG['DRY_RUN'],
)

upsert_dataframe(
    client=sb,
    table_name=TABLES['pred_threshold'],
    df=pred_thr_payload,
    on_conflict='recruit_id,model_version,as_of_date',
    dry_run=CONFIG['DRY_RUN'],
)

upsert_dataframe(
    client=sb,
    table_name=TABLES['factoid_vectors'],
    df=factoid_payload,
    on_conflict='id',
    dry_run=CONFIG['DRY_RUN'],
)

[gi_player_master] rows=41,926, batches=84, on_conflict=recruit_id
[gi_player_master] Upsert complete
[gi_player_link_bridge] rows=25,353, batches=51, on_conflict=recruit_id,sports_ref_id
[gi_player_link_bridge] Upsert complete
[gi_scouting_report_features] rows=41,925, batches=84, on_conflict=recruit_id
[gi_scouting_report_features] Upsert complete
[gi_model_prediction_score] rows=14,392, batches=29, on_conflict=recruit_id,model_version,as_of_date
[gi_model_prediction_score] Upsert complete
[gi_model_prediction_thresholds] rows=14,392, batches=29, on_conflict=recruit_id,model_version,as_of_date
[gi_model_prediction_thresholds] Upsert complete
[gi_factoid_vectors] rows=336, batches=1, on_conflict=id
[gi_factoid_vectors] Upsert complete


## Query Patterns for Agent Workflow

These snippets show how the agent can pull context:
- player + scouting + xgboost score/probabilities via SQL
- factoids via vector search
- optional news chunks + summaries for latest intel

In [27]:
SQL_JOIN_EXAMPLE = f"""
select
  pm.recruit_id,
  pm.sports_ref_id,
  pm.player_name,
  pm.position,
  pm.rating,
  pm.height_raw,
  pm.weight_raw,
  pm.height_inches,
  pm.weight_lbs,
  sf.hs_athletic_background,
  sf.scouting_json,
  ps.predictive_score_0_100,
  pt.contrib_tier_raw,
  pt.prob_ge20,
  pt.prob_ge50,
  pt.prob_ge80
from {TABLES['player_master']} pm
left join {TABLES['scouting_features']} sf on sf.recruit_id = pm.recruit_id
left join {TABLES['pred_score']} ps on ps.recruit_id = pm.recruit_id
left join {TABLES['pred_threshold']} pt on pt.recruit_id = pm.recruit_id
where pm.recruit_id = :recruit_id;
"""

VECTOR_RPC_EXAMPLE = f"""
-- Requires a match function over {TABLES['factoid_vectors']} embedding column
select *
from match_factoids(
  query_embedding => :query_embedding,
  match_threshold => 0.25,
  match_count => 8
);
"""

print('SQL join example:')
print(SQL_JOIN_EXAMPLE)
print('Vector RPC example:')
print(VECTOR_RPC_EXAMPLE)

SQL join example:

select
  pm.recruit_id,
  pm.sports_ref_id,
  pm.player_name,
  pm.position,
  pm.rating,
  pm.height_raw,
  pm.weight_raw,
  pm.height_inches,
  pm.weight_lbs,
  sf.hs_athletic_background,
  sf.scouting_json,
  ps.predictive_score_0_100,
  pt.contrib_tier_raw,
  pt.prob_ge20,
  pt.prob_ge50,
  pt.prob_ge80
from gi_player_master pm
left join gi_scouting_report_features sf on sf.recruit_id = pm.recruit_id
left join gi_model_prediction_score ps on ps.recruit_id = pm.recruit_id
left join gi_model_prediction_thresholds pt on pt.recruit_id = pm.recruit_id
where pm.recruit_id = :recruit_id;

Vector RPC example:

-- Requires a match function over gi_factoid_vectors embedding column
select *
from match_factoids(
  query_embedding => :query_embedding,
  match_threshold => 0.25,
  match_count => 8
);



## Optional: Runtime News Ingestion Hooks

These are placeholders for your live workflow (DuckDuckGo -> Gemini 2.5-lite).

- Use `gi_news_chunks` for retrieved article chunks + embeddings
- Use `gi_news_summaries` for Gemini summaries and extracted stat JSON
- Final scout LLM queries SQL + vector + summaries at inference time

In [28]:
NEWS_INGEST_TEMPLATE = {
    'chunk_id': '<hash_or_uuid>',
    'recruit_id': '<recruit_id>',
    'sports_ref_id': '<sports_ref_id_or_null>',
    'source': 'DuckDuckGo',
    'title': '<article_title>',
    'url': '<article_url>',
    'chunk_text': '<text_chunk>',
    'published_at': None,
    'embedding': [],
    'embedding_model': CONFIG['EMBEDDING_MODEL'],
}

SUMMARY_TEMPLATE = {
    'summary_id': '<hash_or_uuid>',
    'recruit_id': '<recruit_id>',
    'sports_ref_id': '<sports_ref_id_or_null>',
    'llm_model': 'gemini-2.5-flash-lite',
    'prompt_version': 'v1',
    'summary_text': '<summary>',
    'extracted_stats_json': {'key_stats': []},
}

print('News chunk template:')
print(json.dumps(NEWS_INGEST_TEMPLATE, indent=2))
print('\nSummary template:')
print(json.dumps(SUMMARY_TEMPLATE, indent=2))

News chunk template:
{
  "chunk_id": "<hash_or_uuid>",
  "recruit_id": "<recruit_id>",
  "sports_ref_id": "<sports_ref_id_or_null>",
  "source": "DuckDuckGo",
  "title": "<article_title>",
  "url": "<article_url>",
  "chunk_text": "<text_chunk>",
  "published_at": null,
  "embedding": [],
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
}

Summary template:
{
  "summary_id": "<hash_or_uuid>",
  "recruit_id": "<recruit_id>",
  "sports_ref_id": "<sports_ref_id_or_null>",
  "llm_model": "gemini-2.5-flash-lite",
  "prompt_version": "v1",
  "summary_text": "<summary>",
  "extracted_stats_json": {
    "key_stats": []
  }
}
